# Compile the main simulator code using cython 

In [ ]:
!python ../setup.py build_ext --inplace

# Import all the relevant files 

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import importlib
import seaborn as sns
import pathos.multiprocessing
import pickle

In [ ]:
import sys
sys.path.append('../')

#Importing scripts:

#Import relevant frames:
import common.cbgt as cbgt
import common.pipeline_creation as pl_creat

#Import plotting functions:
import common.plotting_functions as plt_func
import common.plotting_helper_functions as plt_help
import common.postprocessing_helpers as post_help

importlib.reload(plt_help)
importlib.reload(plt_func)
importlib.reload(post_help)

%load_ext autoreload
%autoreload 2
%reload_ext autoreload 

import warnings
warnings.simplefilter('ignore', category=FutureWarning)

In [ ]:
def saveresults_vars(variable, prefix):
    pickle.dump(variable, open(prefix, 'wb'))
    
def loadresults_vars(prefix):
    return pickle.load(open(prefix, "rb"))

# Choose the experiment and create the main pipeline

Choose the experiment to run, and define the number of choices:

In [ ]:
#Choose the experiment:
experiment_choice = 'stop-signal'

if experiment_choice == 'stop-signal':
    import stopsignal.paramfile_stopsignal as paramfile
elif experiment_choice == 'n-choice':
    import nchoice.paramfile_nchoice as paramfile

number_of_choices = 1

#Call choose_pipeline with the pipeline object:
pl_creat.choose_pipeline(experiment_choice)
#Create the main pipeline:
pl = pl_creat.create_main_pipeline(runloop=True)

#Set a seed:
seed = np.random.randint(0,99999999,1)[0]

print(seed)

Define data and figures directories:

In [ ]:
data_dir = "../Data/"
figure_dir = "../Figures/"

# Running the pipeline

### Define configuration parameter

In [ ]:
configuration = {
    'experimentchoice': experiment_choice,
    'inter_trial_interval': None,  # default = 600ms
    'thalamic_threshold': 30.,
    'movement_time': ['constant', 300], #default sampled from N(250,1.5), ["constant",250], ["mean",250]
    'choice_timeout': 1000,
    
    'params': paramfile.celldefaults,  #neuron parameters
    'pops': paramfile.popspecific,    #population parameters
    'receps' : paramfile.receptordefaults, #receptor parameters
    'base' : paramfile.basestim,   #baseline stimulation parameters
    'dpmns' : paramfile.dpmndefaults,  #dopamine related parameters
    'd1' : paramfile.d1defaults,     #dSPNs population related parameters
    'd2' : paramfile.d2defaults,     #iSPNs population related parameters
    'channels' : pd.DataFrame([["left"]], columns=['action']), #action channels related parameters (init_params.py)
    'number_of_choices': number_of_choices,
    #'actionchannels' : pd.DataFrame([[1],[2]], columns=['action']), #labels for the actions (init_params.py)
    #'actionchannels' : pd.DataFrame([["left"],["right"]], columns=['action']), #labels for the actions (init_params.py)
    'newpathways' : None, #connectivity parameters (popconstruct.py)
    'Q_support_params': None, #initialization of Q-values update (qvalues.py) 
    'err_support_params': None,
    'Q_df_set': pd.DataFrame([[0.5]],columns=["left"]), #initialized Q-values df (qvalues.py)  
    'err_df_set': pd.DataFrame([[0]],columns=["left"]),
    'n_trials': 300, #number of trials (generateepochs.py)
    'volatility': [None, "exact"], #frequency of changepoints (generateepochs.py)
    'conflict': (1.0), #probability of the preferred choice (generateepochs.py)
    'reward_mu': 1, #mean for the magnitude of the reward (generateepochs.py)
    'reward_std': 0.1, #std for the magnitude of the reward (generateepochs.py)
    'maxstim': 1.00, # amplitude of the cortical input over base line 
    'sustainedfraction': 0.75,
    
    #Stop signal
    'stop_signal_present': [True],
    'stop_signal_probability': [.3],  #probability of trials that will get the stop-signal / list of trial numbers
    'stop_signal_amplitude': [1.],    #amplitude of the stop signal over base line 
    'max_stop_amplitude': [1.], 
    
    'stop_signal_onset_type': "uniform", #"probe" or "uniform" #if fix, put "probe"
    'stop_signal_onset': [[10., 325.]],  # probe [[values]]  
                                         # uniform  [[start, end]]
                                         # early [[mean, std]] 
                                         # late  [[mean, std]]   #before mu = 175
                                         # fix: probe [[value]]
    'stop_signal_duration' : [400.-70.], # duration is internally defined as timeout - stop signal onset 
    'stop_signal_channel': ["all"], #"all" (all channels are given the stop signal) 
                                    #"any" (channel given the stop signal is chosen randomly)
                                    # [list of channels] == subset of channels given the stop signal
    'stop_signal_population':["CxS"],

    'corticostriatal_plasticity_present': True,
    'corticoiSPN_plasticity_present': False,
    
    'record_variables':["weight_go", "weight_stop", "stop_input"],
    'target_RT': 175, #target time 
    'scale_power': 2., 
    'scale_factor_stop': 0.05, 
    'tau_CxS_decay': 3500, 
    
    #Opto signal
    'opt_signal_present': [False],
    'opt_signal_probability': [1.], #probability of trials that will get the optogenetic signal / list of trial numbers
    'opt_signal_amplitude': [.8],   #amplitude of the stop signal over base line
    'opt_signal_onset': [0.],       #in ms
    'opt_signal_duration': [150.],
    'opt_signal_channel': ["all"],
    'opt_signal_population':["Th"],
}

### Run the simulation

ExecutionManager class can take for 'use': 

- 'none', that corresponds to the singlethreaded mode;
- 'pathos', that corresponds to python's multiprocessing mode;
- 'ray', that corresponds to a multiprocessing library for python that operates on a client-server mode.

The default value is None (singlethreaded mode).

In [ ]:
data_dir

In [ ]:
use_library = "pathos" # "none" or "pathos" or "ray"
num_sims = 1
num_cores = 7

In [ ]:
results = cbgt.ExecutionManager(cores=num_cores,use=use_library).run([pl]*num_sims,[configuration]*num_sims)

In [ ]:
datatables = cbgt.collateVariable(results, 'datatables')
datatables_dpmn = cbgt.collateVariable(results, 'datatables_dpmn')

importlib.reload(post_help)
recorded_variables = post_help.extract_recording_variables(results,results[0]['record_variables'], seed)
importlib.reload(plt_help)
firing_rates, rt_dist, summary_df = plt_help.extract_relevant_frames(results, seed, experiment_choice)

In [ ]:
datatables[0]

In [ ]:
recorded_variables['weight_go']

In [ ]:
results[0]['Q_df'][1:]

In [ ]:
results[0]['err_df'][1:]

In [ ]:
summary_df[0]

# Results

List all the agent variables accessible: 

In [ ]:
results[0].keys()

In [ ]:
agent = results[0]['agent']

Extract all the relevant dataframes:

In [ ]:
importlib.reload(plt_help)
firing_rates, rt_dist, summary_df = plt_help.extract_relevant_frames(results, seed, experiment_choice)

In [ ]:
recorded_variables = post_help.extract_recording_variables(results,results[0]['record_variables'],seed)

Plot FRs:

In [ ]:
datatables = cbgt.collateVariable(results,'datatables')
datatables[0]

In [ ]:
import importlib 
importlib.reload(plt_func)

In [ ]:
FR_fig_handles = plt_func.plot_fr_new(firing_rates,datatables,results,experiment_choice,True)
FR_fig_handles[0].savefig(figure_dir+"FR_Variant2.png",dpi=300)